In [1]:
!wget -q https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt -O tinyshakespeare.txt

In [2]:
from torch import nn

class Encoder:

    def __init__(self, path):
        with open(path) as f:
            lines = f.readlines()

        text = "\n".join(lines)

        self.dictionary = {}

        index = 1
        self.dictionary = {"⒨": 0}
        for char in text:
            if self.dictionary.get(char):
                continue

            self.dictionary[char] = index
            index += 1

    def encode(self, text):
        return [self.dictionary[char] for char in text]

    def decode(self, encoded) -> list[int]:
        inv_dict = {v: k for k, v in self.dictionary.items()}
        return [inv_dict[encoding] for encoding in encoded]

    def vocab(self):
        return self.dictionary.keys()

encoder = Encoder("tinyshakespeare.txt")
encoded = encoder.encode("Hello World!")
print(encoded)
decoded = encoder.decode(encoded)
print(decoded)

[50, 9, 29, 29, 15, 6, 36, 15, 3, 29, 19, 44]
['H', 'e', 'l', 'l', 'o', ' ', 'W', 'o', 'r', 'l', 'd', '!']


In [3]:
from torch import nn, Tensor, softmax, randn

class Transformer(nn.Module):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

        self.encoder = Encoder("tinyshakespeare.txt")

        vocab_size = len(self.encoder.vocab())
        max_seq_len = 1024
        embed_dim = 256
        hidden_dim = 4 * embed_dim
        num_layers = 16

        self.token_emb = nn.Parameter(randn(vocab_size, embed_dim))
        self.pos_emb = nn.Parameter(randn(max_seq_len, embed_dim))
        self.mask_prob_emb = nn.Linear(1, embed_dim)

        self.layers = nn.ModuleList([
            nn.ModuleList([
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, embed_dim), #W_Q
                nn.Linear(embed_dim, embed_dim), #W_K
                nn.Linear(embed_dim, embed_dim), #W_V

                #FFN
                nn.LayerNorm(embed_dim),
                nn.Linear(embed_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, embed_dim)])
            for _ in range(num_layers)
        ])

        self.output = nn.Linear(embed_dim, vocab_size)

    def embedding(self, x: Tensor, mask_prob: Tensor):
        t_emb = self.mask_prob_emb(mask_prob)
        return self.token_emb[x] + self.pos_emb[:x.shape[-1]] + t_emb

    def attention(self, Q: Tensor, K: Tensor, V: Tensor):
        d_k = K.shape[-1]
        return softmax(Q @ K.transpose(-2, -1) / (d_k ** 0.5), dim=-1) @ V

    def forward(self, x: Tensor, mask_prob: Tensor) -> Tensor:
        x = self.embedding(x, mask_prob)
        for ln1, W_Q, W_K, W_V, ln2, linear1, relu, linear2 in self.layers:
            x = x + self.attention(W_Q(ln1(x)), W_K(ln1(x)), W_V(ln1(x)))
            x = x + linear2(relu(linear1(ln2(x))))

        return self.output(x)

In [4]:
import random
import torch
import time
import ipywidgets as widgets
from IPython.display import display

def sample(model, query, length, device, total_steps=20):

    # Widget for printing 
    out = widgets.Output()
    display(out)

    # Tokenize text input
    tokens = model.encoder.encode(query)
    x = torch.zeros(length, dtype=torch.long, device=device)
    x[:len(tokens)] = torch.tensor(tokens, device=device)
    fixed = (x != 0)

    with torch.no_grad():
        for step in range(total_steps):
            
            # Predict probabilites for each token at each position
            mask_prob = torch.tensor([1.0 - step / total_steps], device=device)
            predictions = model.forward(x, mask_prob)
            probs = torch.softmax(predictions, dim=-1)

            # Get positions of token set to the special 
            mask_positions = (x == 0) & ~fixed
            if not mask_positions.any():
                break

            # Iterate overall all masked tokens
            for pos in mask_positions.nonzero():
                # Randomly unmask tokens
                if random.random() < 1 / (total_steps - step):
                    # Select likely token
                    x[pos] = torch.multinomial(probs[pos], 1)
            
            # Print out the generated text
            with out:
                out.clear_output(wait=True)
                print(''.join(model.encoder.decode(x.tolist())))



## Training

1. Sample chunk of text data: $x_0 \sim TinyShakespeare$
2. Sample masking probability for each token: $t_{token} \sim Uniform(0,1)$
3. Add noise as masking for each token: $x_{t,token}=x_{0,token} (1 - m)$, where $m \sim Bernoulli(t_{tokem})$
4. Run diffusion model $D_{\theta}$ with masked tokens: ${\hat x_0} = D_{\theta}(x_{t},t_{token})$
5. The prediction $\hat x_{0}$ is a categorical distribution for each token. Use cross entropy to train the model for each token: $L = -Σ ~x_{0,token}~log(\hat x_{0,token})$
6. Backprop

In [5]:
import math
import torch
import random
import time
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with open("tinyshakespeare.txt") as f:
    text = f.read()

split = int(0.9 * len(text))
train_text, val_text = text[:split], text[split:]

seq_len = 128
batch_size = 64
iterations = 100000
checkpoint_every = 10000

transformer = Transformer().to(device)
optimizer = torch.optim.Adam(transformer.parameters(), lr=1e-4)

def grab_chunk(src) -> torch.Tensor:
    start = random.randint(0, len(src) - seq_len)
    return torch.tensor(transformer.encoder.encode(src[start:start + seq_len]), device=device)

def add_noise(x, t):
    mask = (torch.rand_like(x, dtype=torch.float) < t).long()
    return x * (1 - mask), mask

In [6]:
for i in tqdm(range(iterations)):
    batch = torch.stack([grab_chunk(train_text) for _ in range(batch_size)])
    t = random.uniform(0, 1)
    masked, mask = add_noise(batch, t)
    preds = transformer(masked, torch.tensor([t], device=device))
    loss = torch.nn.functional.cross_entropy(preds[mask == 1], batch[mask == 1])

    if i % checkpoint_every == 0:
        sample(transformer, "To be, ", 64, device)
    
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

# Save final model
sample(transformer, "To be, ", 64, device)
torch.save(transformer.state_dict(), f"mdlm.pt")

  0%|          | 0/100000 [00:00<?, ?it/s]

Output()

 10%|█         | 10000/100000 [20:39<3:04:39,  8.12it/s]

Output()

 20%|██        | 20000/100000 [40:57<2:36:28,  8.52it/s]

...

Output()

 90%|█████████ | 90000/100000 [2:54:48<19:00,  8.77it/s]

Output()

100%|██████████| 100000/100000 [3:13:56<00:00,  8.59it/s]


Output()

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer = Transformer().to(device)
transformer.load_state_dict(torch.load("mdlm.pt"))

/tmp/ipykernel_3434924/3027920531.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  transformer.load_state_dict(torch.load("mdlm.pt"))


<All keys matched successfully>

In [10]:
sample(transformer, "To be, ", 128, device, 1000)

Output()

In [9]:
def count_params(model):
    return sum(p.numel() for p in model.parameters())

count_params(transformer)

11880002